In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge,Lasso,LinearRegression
from sklearn.metrics import r2_score

df = pd.read_csv('50_Startups.csv')

print(df.head())

print(df.info())

   R&D Spend  Administration  Marketing Spend       State     Profit
0  165349.20       136897.80        471784.10    New York  192261.83
1  162597.70       151377.59        443898.53  California  191792.06
2  153441.51       101145.55        407934.54     Florida  191050.39
3  144372.41       118671.85        383199.62    New York  182901.99
4  142107.34        91391.77        366168.42     Florida  166187.94
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB
None


In [3]:
df = pd.get_dummies(df, columns=['State'], drop_first=True, dtype=int)

x = df.drop('Profit', axis=1)
y = df['Profit']   

print(x.head())

   R&D Spend  Administration  Marketing Spend  State_Florida  State_New York
0  165349.20       136897.80        471784.10              0               1
1  162597.70       151377.59        443898.53              0               0
2  153441.51       101145.55        407934.54              1               0
3  144372.41       118671.85        383199.62              0               1
4  142107.34        91391.77        366168.42              1               0


In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)



[[ 0.34202149  0.22787678  0.12425038  1.36277029 -0.69388867]
 [ 1.36207849 -1.0974737   1.14990688  1.36277029 -0.69388867]
 [-0.71081297 -2.5770186  -0.34136825 -0.73379939 -0.69388867]
 [ 0.90611438  1.0172367   0.66890185 -0.73379939  1.44115338]
 [ 1.40997088 -0.09115403  1.30006861 -0.73379939  1.44115338]]


In [9]:
linear_model = LinearRegression()

linear_model.fit(x_train,y_train)

y_pred = linear_model.predict(x_test)

print("R2 score for Linear Regression:",r2_score(y_test,y_pred))

R2 score for Linear Regression: 0.8987266414319839


In [11]:
ridge_parama = {'alpha':[0.001,0.01,0.1,1,10,100,1000]}

ridge_model = Ridge()
ridge_cv = GridSearchCV(ridge_model,ridge_parama,scoring='r2',cv=5)

ridge_cv.fit(x_train,y_train)

print("Best alpha for Ridge Regression:",ridge_cv.best_params_)
ridge_pred = ridge_cv.predict(x_test)
print("R2 score for Ridge Regression:",r2_score(y_test,ridge_pred))

Best alpha for Ridge Regression: {'alpha': 0.1}
R2 score for Ridge Regression: 0.898537985100537


In [12]:
lasso_parama = {'alpha':[0.001,0.01,0.1,1,10,100,1000]}
lasso_model = Lasso()

lasso_cv = GridSearchCV(lasso_model,lasso_parama,scoring='r2',cv=5)
lasso_cv.fit(x_train,y_train)

print("Best alpha for Lasso Regression:",lasso_cv.best_params_)
lasso_pred = lasso_cv.predict(x_test)
print("R2 score for Lasso Regression:",r2_score(y_test,lasso_pred))

Best alpha for Lasso Regression: {'alpha': 1000}
R2 score for Lasso Regression: 0.9166222432779318


In [13]:
print("Lasso Coefficients:",lasso_cv.best_estimator_.coef_)

Lasso Coefficients: [36824.84457828  -654.78829054  3551.90737318     0.
    -0.        ]


In [14]:
import pickle

# સ્કેલર (સ્કેલિંગ કરવા માટે) અને મોડેલ (પ્રેડિક્શન માટે) બંનેને સેવ કરો
pickle.dump(scaler, open('scaler.pkl', 'wb'))
pickle.dump(lasso_cv.best_estimator_, open('model.pkl', 'wb'))

print("મોડેલ અને સ્કેલર સફળતાપૂર્વક સેવ થઈ ગયા છે!")

મોડેલ અને સ્કેલર સફળતાપૂર્વક સેવ થઈ ગયા છે!
